In [35]:
import unicodedata

from openai import OpenAI

API_KEY = 'sk-jwx9HLq3fDya9Sh4140eFf5cC4D94d4c80963229Ad93018b'

client = OpenAI(
    api_key=API_KEY,
    base_url="https://lonlie.plus7.plus/v1"
)

In [39]:
import os
import json
import random
from tqdm import tqdm
from datetime import datetime


version = 'v4'
datapath = f'../filtered_news'
n_qa = 1
n_resp = 10


def get_qa_from_openai():
    news_contents = {}
    jsons = [file for file in os.listdir(datapath) if file.endswith('.json') and '_qa' not in file]
    # random.shuffle(jsons)
    for news in jsons:
        news_name = news.split('.')[0]
        news_contents[news_name] = json.load(open(os.path.join(datapath, news)))
    id = 1
    for name, news in tqdm(news_contents.items()):
        if os.path.exists(os.path.join(datapath, f'{name}_qa.json')):
            print(f'{name}_qa.json exists, skip...')
            continue
        news_body = news['body']
        
        user_prompt_versions = {
            'v1': "请针对如下新闻文本，提出五个有意义的、相关的问题，并给出相应的回答。每组问答格式如下：\n问：{问题}\n答：{回答}\n不要编号。\n新闻如下：\n",
            'v2': "请针对如下新闻文本，提出五个有意义的、相关的问题以及相应回答。这些问题应该包含必要的新闻原文背景，使得在不提供新闻文本时，也能得到同样的回答。每组问答格式如下：\n问：{问题}\n答：{回答}\n不要编号。\n新闻如下：\n",
            'v3': "请针对如下新闻文本，提出五个具体问题，每个问题都要详尽给出必要背景（如时间、地点、事件、活动名称、相关人物等），并给出每个问题的相应回答。每组问答格式如下：\n问：{带有详细背景的问题}\n答：{回答}\n不要编号。\n新闻如下：\n",
            'v4.5': "请针对如下新闻文本，提出一个具体问题，问题要给出详尽必要的背景（如时间、地点、事件、活动名称、相关人物等），并给出相应回答。问答格式如下：\n问：{带有详细背景的问题}\n答：{回答}\n新闻如下：\n",
            'v4': "请针对如下新闻文本，提出一个具体问题，问题要包含详尽必要的背景，并给出回答。格式如下：\n问：{带有完整背景的问题}\n答：{回答}\n不要输出多余内容。\n新闻如下：\n",
            'v5': "请针对如下新闻文本，提出五个具体问题，问题要包含详尽完整的背景，不同问题的背景可以重复，至少要给出具体的时间、地点、事件、活动名称、会议名称、相关单位和人员等，并给出每个问题的相应回答。每组问答格式如下：\n问：{带有完整背景的问题}\n答：{回答}\n不要编号。\n新闻如下：\n",
        }
        user_prompt = user_prompt_versions[version]
        print('第 %d 条新闻' % id)
        id += 1
        print('<<<\n'+user_prompt+news_body)  
        
        qa = []
        for _ in range(n_qa):
            completion = client.chat.completions.create(
                messages=[
                    # {"role": "system", "content": "You are a helpful assistant. User is asking for some possible questions to ask about the news, you should generate some questions based on the news alongwith the answers. The questions should be open-ended and should not be too specific."},
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": user_prompt+news_body}
                ],
                model="gpt-3.5-turbo-1106",
                n=n_resp,
                top_p=0.95,
                temperature=2.0,
            )
            for i in range(n_resp):
                message = completion.choices[i].message
                content = unicodedata.normalize('NFKC', message.content)
                qa.append(content)
                print('>>>\n'+content)
        
        data = news
        # data['prompt'] = user_prompt
        data['qa'] = qa
        timestamp = datetime.now().strftime('%Y%m%d%H%M%S')
        json.dump(data, open(os.path.join(datapath, f'{name}_qa.json'), 'w'), ensure_ascii=False, indent=4)
        # print('>>>\n'+qa)
        print('---\n\n')
        # break
        

get_qa_from_openai()
# $0.0826880
# $0.0868800 single
# $0.1075380 *10

  0%|          | 0/3297 [00:00<?, ?it/s]

c1247a104787_qa.json exists, skip...
c1247a105658_qa.json exists, skip...
c1247a107683_qa.json exists, skip...
c1247a131533_qa.json exists, skip...
c1247a107820_qa.json exists, skip...
c1247a104562_qa.json exists, skip...
第 1 条新闻
<<<
请针对如下新闻文本，提出一个具体问题，问题要包含详尽必要的背景，并给出回答。格式如下：
问：{带有完整背景的问题}
答：{回答}
不要输出多余内容。
新闻如下：
日前，复旦大学附属肿瘤医院乳腺外科教授邵志敏、江一舟团队发布的一项前沿研究成果，发现了三阴性乳腺癌转移的一条新路径：“三阴性乳腺癌细胞的‘代谢基因PDSS1’通过其代谢途径，影响下游信号，经过一系列信息的加工处理，最终促使了癌细胞的转移”。这项突破性的发现，意味着基于三阴性乳腺癌转移路径的精准治疗将成为可能。8月18日，该项成果在线发表于国际著名期刊《癌症研究》（CancerResearch）。缺乏精准治疗“靶点”：三阴性乳腺癌疗效不佳乳腺癌是女性恶性肿瘤排行榜的“头号杀手”，而三阴性乳腺癌是乳腺癌的一种亚型，约占总体乳腺癌人群的15%。由于恶性程度高且转移复发风险高，三阴性乳腺癌素有“最毒”乳腺癌之称。据介绍，雌激素受体、孕激素受体、HER2是当前乳腺癌治疗最具有临床意义的三种细胞分子，三阴性乳腺癌之所以被称为“三阴”，正是因为在三阴性乳腺癌细胞中这三个重要分子均表达为阴性。邵志敏表示：“在乳腺癌的治疗中，雌激素受体、孕激素受体、HER2这三种细胞分子常常被作为精准靶向治疗的‘靶标’，即通常所说的靶点。但目前三阴性乳腺癌缺乏明确的‘靶标’，也导致相应的靶向治疗药物缺乏，使得三阴性乳腺癌的疗效要明显低于其他亚型的乳腺癌。”基于全球最大三阴性乳腺癌基因图谱：“锁定”促进三阴性乳腺癌转移关键分子既往数据表明，三阴性乳腺癌在治疗后2-3年左右会达到转移风险的高峰，但其转移路径不详的难题一直困惑着医学专家。“鉴于三阴性乳腺癌转移性远高于其他类型的乳腺癌，我们思考，如果从三阴性乳腺癌的转移机制中探索其独特性，分析其发生转移的内在联系，就很有可能在这个过程中发

  0%|          | 7/3297 [00:14<1:52:24,  2.05s/it]

>>>
问:三阴性乳腺癌转移机制长期困扰医学界,且目前缺乏明确的精准治疗“靶标”,这一突破性的新研究发现了一个影响癌细胞转移的新路径。可以具体介绍一下该突破性发现对于三阴性乳腺癌治疗和精准医学的意义是什么?同时,这项发现未来如何为患者带来实际的临床治疗效益?
答:该项突破性研究发现了三阴性乳腺癌转移的新机制,其中代谢基因PDSS1通过调控相关信号通路,推动了癌细胞的转移。这一发现对三阴性乳腺癌的治疗和精准医学意义重大,因为目前该乳腺癌亚型缺乏明确的精准治疗靶标,且其疗效低于其他亚型。基于该研究成果,可以有望开发针对PDSS1等相关信号通路的精准靶向治疗药物,提高治疗效果,并降低患者的转移风险。因此,这项发现对患者未来的临床治疗有望带来积极的效益,为提供更个体化、精准的治疗方案提供了新的可能性。
>>>
问:什么是三阴性乳腺癌?目前三阴性乳腺癌存在什么样的治疗难题?邵志敏团队最新研究有何重大突破?
答:三阴性乳腺癌是乳腺癌的一种亚型,约占总体乳腺癌人群的15%,因雌激素受体、孕激素受体、HER2均为阴性而得名。目前三阴性乳腺癌缺乏明确的治疗靶标和相应的靶向治疗药物,使得其疗效明显低于其他乳腺癌亚型。邵志敏团队最新研究发现了代谢基因PDSS1在三阴性乳腺癌转移中的重要作用,并锁定了其下游信号通路PDSS1/CAMK2A/STAT3,为基于三阴性乳腺癌转移路径的精准靶向治疗提供了可能。
>>>
问:基于邵志敏、江一舟团队的研究成果,关于三阴性乳腺癌转移路径的发现有何突破性意义?是否能够为该病型提供精准治疗的新靶点?
答:邵志敏、江一舟团队发现了三阴性乳腺癌转移的一条新路径,并锁定了促进该病型转移的关键分子PDSS1。这项突破性发现意味着基于三阴性乳腺癌转移路径的精准治疗将成为可能。该发现为三阴性乳腺癌提供了潜在的精准治疗靶点,使得针对该病型的治疗将迈向更具针对性和有效性。
>>>
问:代谢基因PDSS1如何影响三阴性乳腺癌细胞的转移?在研究中是如何揭示其复杂作用和相关机制的?
答:代谢基因PDSS1催化细胞产生一种辅酶代谢物,而这种代谢物对于细胞基因信息的调控和传导发挥作用。进一步研究发现,代谢基因PDSS1通过调控癌基因STAT3来影响癌细胞转移,使得其信息转移至细胞核,驱动癌细胞转移。研究团队锁定了代谢通路PDSS1/CAMK2A/STAT3,并指出该信号通路有望成为

  0%|          | 8/3297 [00:24<2:45:32,  3.02s/it]

>>>
问:中国复旦大学举办了2019复旦·思源全球领袖论坛,论坛主题为“科技创新融合共赢”,邀请了来自全球各地的政界、商界、学界等领域的精英领袖参与,并且涉及了诸如创业、教育和科技发展等多方面话题。请问这次论坛主要呈现了哪些核心观点,以及它对中国和全球的科技创新和合作发展有何影响? 
答:2019复旦·思源全球领袖论坛主要呈现了诸如创业、教育、科技发展等方面的核心观点。罗康瑞先生强调了创业者需要了解市场、管理现金流、保持初心等关键要素。吴家玮先生则强调了科技创新需要适宜的文化环境和博雅教育的重要性。汪建先生指出生命科学和基因科技将推动未来的产业发展。加里·瑞斯彻先生谈到中国市场在医疗健康和TMT产业中的迅速增长。这些核心观点突出了创业、教育、科技领域的重要性和发展方向,对中国和全球科技创新与合作发展有着深远的影响。
>>>
问:中国在经济转型升级过程中,商业创新和技术创新的双轮驱动如何提升中国的竞争力?论坛中哪些演讲内容能够提供关于此问题的洞见?

答:商业创新和技术创新的双轮驱动是中国获得长久竞争力并推动全球经济融合和持续发展的关键。在本次复旦·思源全球领袖论坛中,罗康瑞先生提到了初创阶段创业者应如何了解市场、客观评估自身,管理好现金流;在扩张阶段如何在扩大规模和保持初心中保持平衡,做好管理架构,强调了商业创新在企业增值中的重要性。另外,汪建先生从生命科学和基因科技角度强调了满足人类对生命需求的生命科学研究对产业发展的重要作用。以上观点给出了商业创新和技术创新对提升中国竞争力的洞见。
>>>
问:2019复旦·思源全球领袖论坛的主题是什么,论坛汇聚了哪些领袖人物?主要嘉宾分别发表了哪些主题演讲?他们对未来的发展有何建议?

答:2019复旦·思源全球领袖论坛的主题是“科技创新融合共赢”,论坛汇聚了来自全球各地政界、商界、学界等领域的精英领袖。主要嘉宾包括瑞安集团主席罗康瑞先生、香港科技大学创校校长吴家玮先生、华大基因董事长汪建先生、启明创投创始主管合伙人加里·瑞斯彻先生。他们分别发表了“创业者的角色”、"博雅教育:科技创新的基石"、“World-class Business=World-class Science: Venture Capital or Venture Science/Tech?”和“Identify potential early. Partn

KeyboardInterrupt: 